## Logistic Regression
### Problem: Determine the likelihood of an employee being promoted based on their service statistics.

### Table of Contents
* [Importing Dependencies](#idp)
* [Importing Dataset](#ids)
* [Analysing Data](#ad)
* [Optimizing Data](#od)
* [Training and testing the model](#ttm)
* [Using the model](#utm)

### Importing Dependencies <a id = "idp">

In [105]:
#Packages
import numpy as np
import pandas as pd
import re

#Models & Encoding
from sklearn.linear_model import LogisticRegression
from category_encoders import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

#Success Metrics
from sklearn.metrics import classification_report

%matplotlib inline

### Importing Dataset <a id = "ids">

In [15]:
df = pd.read_csv('hr_data.csv')
df

,employee_id,department,region,education,gender,recruitment_channel,no_of_trainings,age,previous_year_rating,length_of_service,KPIs_met >80%,awards_won?,avg_training_score,is_promoted
0,65438,Sales & Marketing,region_7,Master's & above,f,sourcing,1,35,5.0,8,1,0,49,0
1,65141,Operations,region_22,Bachelor's,m,other,1,30,5.0,4,0,0,60,0
2,7513,Sales & Marketing,region_19,Bachelor's,m,sourcing,1,34,3.0,7,0,0,50,0
3,2542,Sales & Marketing,region_23,Bachelor's,m,other,2,39,1.0,10,0,0,50,0
4,48945,Technology,region_26,Bachelor's,m,other,1,45,3.0,2,0,0,73,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54803,3030,Technology,region_14,Bachelor's,m,sourcing,1,48,3.0,17,0,0,78,0
54804,74592,Operations,region_27,Master's & above,f,other,1,37,2.0,6,0,0,56,0
54805,13918,Analytics,region_1,Bachelor's,m,other,1,27,5.0,3,1,0,79,0
54806,13614,Sales & Marketing,region_9,NaN,m,sourcing,1,29,1.0,2,0,0,45,0


### Analysing the data <a id = "ad">

In [9]:
#Checking for null values
df.isnull().sum()

employee_id                0
department                 0
region                     0
education               2409
gender                     0
recruitment_channel        0
no_of_trainings            0
age                        0
previous_year_rating    4124
length_of_service          0
KPIs_met >80%              0
awards_won?                0
avg_training_score         0
is_promoted                0
dtype: int64

In [10]:
#Checking number of 'is_promoted' == true/false
promoted = (df['is_promoted'] == 1).sum()
print('Number of promoted people:',promoted)
not_promoted = (df['is_promoted'] == 0).sum()
print('Number of people NOT promoted',not_promoted)

Number of promoted people: 4668
Number of people NOT promoted 50140


In [14]:
#Checking range of values for numerical columns
df.describe()

,employee_id,no_of_trainings,age,previous_year_rating,length_of_service,KPIs_met >80%,awards_won?,avg_training_score,is_promoted
count,54808.000000,54808.000000,54808.000000,50684.000000,54808.000000,54808.000000,54808.000000,54808.000000,54808.000000
mean,39195.830627,1.253011,34.803915,3.329256,5.865512,0.351974,0.023172,63.386750,0.085170
std,22586.581449,0.609264,7.660169,1.259993,4.265094,0.477590,0.150450,13.371559,0.279137
min,1.000000,1.000000,20.000000,1.000000,1.000000,0.000000,0.000000,39.000000,0.000000
25%,19669.750000,1.000000,29.000000,3.000000,3.000000,0.000000,0.000000,51.000000,0.000000
50%,39225.500000,1.000000,33.000000,3.000000,5.000000,0.000000,0.000000,60.000000,0.000000
75%,58730.500000,1.000000,39.000000,4.000000,7.000000,1.000000,0.000000,76.000000,0.000000
max,78298.000000,10.000000,60.000000,5.000000,37.000000,1.000000,1.000000,99.000000,1.000000


In [116]:
# Checking column data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54808 entries, 0 to 54807
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   employee_id           54808 non-null  int64  
 1   department            54808 non-null  object 
 2   region                54808 non-null  object 
 3   education             52399 non-null  object 
 4   gender                54808 non-null  object 
 5   recruitment_channel   54808 non-null  object 
 6   no_of_trainings       54808 non-null  int64  
 7   age                   54808 non-null  int64  
 8   previous_year_rating  50684 non-null  float64
 9   length_of_service     54808 non-null  int64  
 10  KPIs_met >80%         54808 non-null  int64  
 11  awards_won?           54808 non-null  int64  
 12  avg_training_score    54808 non-null  int64  
 13  is_promoted           54808 non-null  int64  
dtypes: float64(1), int64(8), object(5)
memory usage: 5.9+ MB


Categorical Columns: department, region, education, gender, recruitment channel

Numerical Columns: employee id, no of trainings, age, previous year rating, length of service, KPIs met > 80%, awards won, avg training score

### Optimizing Data <a id = "od">

In [55]:
#Drop redundant column
hr_df = df.drop(['employee_id'], axis = 1)
hr_df

,department,region,education,gender,recruitment_channel,no_of_trainings,age,previous_year_rating,length_of_service,KPIs_met >80%,awards_won?,avg_training_score,is_promoted
0,Sales & Marketing,region_7,Master's & above,f,sourcing,1,35,5.0,8,1,0,49,0
1,Operations,region_22,Bachelor's,m,other,1,30,5.0,4,0,0,60,0
2,Sales & Marketing,region_19,Bachelor's,m,sourcing,1,34,3.0,7,0,0,50,0
3,Sales & Marketing,region_23,Bachelor's,m,other,2,39,1.0,10,0,0,50,0
4,Technology,region_26,Bachelor's,m,other,1,45,3.0,2,0,0,73,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
54803,Technology,region_14,Bachelor's,m,sourcing,1,48,3.0,17,0,0,78,0
54804,Operations,region_27,Master's & above,f,other,1,37,2.0,6,0,0,56,0
54805,Analytics,region_1,Bachelor's,m,other,1,27,5.0,3,1,0,79,0
54806,Sales & Marketing,region_9,NaN,m,sourcing,1,29,1.0,2,0,0,45,0


#### Strategy for data transformation
* Fill null in 'education' with mode, 'previous_year_rating' with median.
* One-Hot Encoding for ''recruitment_channel', 'gender' and 'department' due to low cardinality and unrelated values.
* Ordinal Encoding for 'education' due to related values.
* Target Encoding for 'region' due to high cardinality and unrelated values.
* Standardization due to different ranges in values.


Pipeline used to prevent data leakage and potential for improvement through GridSearch

In [85]:
# Grouping the columns based on the transformations needed
ordinal = ['education']
ohe = ['department', 'recruitment_channel', 'gender']
target = ['region']
numerical = ['no_of_trainings', 'age', 'previous_year_rating', 'length_of_service', 'avg_training_score']

In [86]:
# Transformer for Ordinal
ord_trf = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('ord', OrdinalEncoder(categories=[["Below Secondary", "Bachelor's", "Master's & above"]]))
])

In [87]:
# Transformer for Numerical
num_trf = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'median')),
    ('scaler', StandardScaler())
])

In [88]:
# ColumnTransformer to apply different pipelines simultaneously
preprocessor = ColumnTransformer(
    transformers = [
        ('ord_trf', ord_trf, ordinal),
        ('ohe_trf', OneHotEncoder(handle_unknown='ignore'), ohe),
        ('targ_trf', TargetEncoder(), target),
        ('num_trf', num_trf, numerical)
    ],
    remainder = 'passthrough'
)

In [108]:
# Creating the final pipeline
pipeline = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter = 1000))
])

### Training and Testing the Model <a id = "ttm">

In [109]:
# Split the dataset into train and test sets
x_df = hr_df.drop('is_promoted',axis=1)
y_df = hr_df["is_promoted"]
X_train, X_test, y_train, y_test = train_test_split(x_df, y_df, test_size=0.2, random_state=1,stratify=y_df)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(43846, 12) (10962, 12) (43846,) (10962,)


In [110]:
# Applying the pipeline to train-test splits.
pipeline.fit(X_train, y_train)

train_pred = pipeline.predict(X_train)
test_pred = pipeline.predict(X_test)

In [111]:
# Displays the accuracy of predictions through different success metrics
print(classification_report(y_test, test_pred))

              precision    recall  f1-score   support

           0       0.94      1.00      0.96     10028
           1       0.87      0.26      0.40       934

    accuracy                           0.93     10962
   macro avg       0.90      0.63      0.68     10962
weighted avg       0.93      0.93      0.92     10962



Precision: How many actually deserved the promotion from those predicted?

Recall: What was the percentage of employees who deserved the promotion that I found?

F1-Score: What is the balance between precision and recall?

### Using the model <a id = "utm">

In [ ]:
Operations	region_22	Bachelor's	m	other	1	30	5.0	4	0	0	60	0

In [114]:
# Defining the employee data
employee = pd.DataFrame([{
    'department': 'Operations',
    'region': 'region_22',
    'education': "Bachelor's",
    'gender': 'm',
    'recruitment_channel': 'other',
    'no_of_trainings': 1,
    'age': 30,
    'previous_year_rating': 5.0,
    'length_of_service': 4,
    'KPIs_met >80%': 1,
    'awards_won?': 1,
    'avg_training_score': 80
}])

In [115]:
# Obtain the prediction and Confidence Score
prediction = pipeline.predict(employee)
print("Prediction:", prediction)

probability = pipeline.predict_proba(employee)
print(f"Probability of Promotion: {probability[0][1]:.2%}")

Prediction: [1]
Probability of Promotion: 99.64%
